# MoodMatch.Ai — Train on Colab GPU

Runs the MobileNetV3-Large transfer-learning + fine-tuning pipeline on a free Colab GPU instead of your laptop.

**Before running:** upload `moodmatch_code.zip` and `moodmatch_data.zip` to a Google Drive folder (e.g. `My Drive/MoodMatch.Ai/`), and make sure **Runtime → Change runtime type → T4 GPU** is selected.

Colab sessions are ephemeral — anything in `/content` is deleted when the runtime disconnects. This notebook copies your final checkpoints back to Drive at the end so you don't lose them.

In [ ]:
# Confirm a GPU is attached (should print an NVIDIA T4/other GPU, not "no devices found")
!nvidia-smi

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT this if you used a different Drive folder name
DRIVE_DIR = '/content/drive/MyDrive/MoodMatch.Ai'

## 2. Unpack code + dataset into the Colab VM

In [ ]:
import os

PROJECT_DIR = '/content/MoodMatch.Ai'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Code goes straight into the project root
!unzip -q "{DRIVE_DIR}/moodmatch_code.zip" -d "{PROJECT_DIR}"

# Dataset goes into data/raw/<class>/
os.makedirs(f'{PROJECT_DIR}/data/raw', exist_ok=True)
!unzip -q "{DRIVE_DIR}/moodmatch_data.zip" -d "{PROJECT_DIR}/data/raw"

print('Classes found:')
!ls "{PROJECT_DIR}/data/raw"

## 3. Install dependencies
(Colab already ships torch/torchvision with CUDA — this just adds anything missing.)

In [ ]:
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

## 4. Stage 1 — transfer learning (frozen backbone)

This should now print `Using device: CUDA` (config.py auto-detects it) instead of CPU.

In [ ]:
%cd {PROJECT_DIR}
!python -m training.train

## 5. Stage 2 — fine-tuning (unfreeze later layers)

In [ ]:
%cd {PROJECT_DIR}
!python -m training.fine_tune

## 6. Evaluate on the test set

In [ ]:
%cd {PROJECT_DIR}
!python -m training.evaluate

## 7. Save results back to Drive

**Do this before your Colab session ends** — `/content` is wiped on disconnect, Drive is not.

In [ ]:
import shutil

shutil.make_archive('/content/moodmatch_checkpoints', 'zip', f'{PROJECT_DIR}/models/checkpoints')
!cp /content/moodmatch_checkpoints.zip "{DRIVE_DIR}/"
print('Saved checkpoints to', DRIVE_DIR)

## 8. Bring results back to your laptop

Download `moodmatch_checkpoints.zip` from your Drive `MoodMatch.Ai/` folder, then locally:

```bash
unzip ~/Downloads/moodmatch_checkpoints.zip -d /Users/javid-67239/mdj/MoodMatch.Ai/models/checkpoints
```

Then `python -m training.predict path/to/image.jpg` works locally on the Colab-trained model, no GPU required for inference.